<a href="https://colab.research.google.com/github/legallyananyar-dev/digital-twin-challenge-/blob/main/WarfarinTwin_01_starting_dose_IWPC_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WarfarinTwin, Notebook 1: Starting-Dose Model (IWPC)

## 1. Setup

In [ ]:
!pip -q install xlrd

In [ ]:
import os, glob, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

## 2. Load the IWPC file

In [ ]:
# --- Option B (uncomment and edit) ---
# !git clone https://github.com/legallyananyar-dev/<your-repo>.git
# DATA_PATH = "/content/https://github.com/legallyananyar-dev/digital-twin-challenge-/data/iwpc/PS206767-553247439.xls"

DATA_PATH = globals().get("DATA_PATH", None)
if DATA_PATH is None:
    found = glob.glob("*.xls") + glob.glob("/content/*.xls")
    if found:
        DATA_PATH = found[0]
    else:
        from google.colab import files
        print("Upload the IWPC .xls file (PS206767-553247439.xls)")
        up = files.upload()
        DATA_PATH = list(up.keys())[0]

raw = pd.read_excel(DATA_PATH, sheet_name="Subject Data", engine="xlrd")
print("Loaded:", DATA_PATH, raw.shape)

## 3. Clean the data

In [ ]:
def clean_iwpc(raw):
    d = pd.DataFrame(index=raw.index)
    d["age_decade"] = raw["Age"].astype(str).str.extract(r"(\d+)")[0].astype(float) // 10
    d["height_cm"] = pd.to_numeric(raw["Height (cm)"], errors="coerce")
    d["weight_kg"] = pd.to_numeric(raw["Weight (kg)"], errors="coerce")
    d["male"] = raw["Gender"].map({"male": 1.0, "female": 0.0})

    race = raw["Race (OMB)"].astype(str)
    d["race"] = np.select(
        [race.str.contains("Asian", case=False), race.str.contains("Black", case=False), race.str.contains("White", case=False)],
        ["Asian", "Black", "White"], default="Unknown")

    inducers = raw[["Carbamazepine (Tegretol)", "Phenytoin (Dilantin)", "Rifampin or Rifampicin"]]
    d["inducer"] = inducers.fillna(0).max(axis=1)
    d["amiodarone"] = pd.to_numeric(raw["Amiodarone (Cordarone)"], errors="coerce").fillna(0)

    cyp_ok = ["*1/*1", "*1/*2", "*1/*3", "*2/*2", "*2/*3", "*3/*3"]
    d["cyp2c9"] = raw["CYP2C9 consensus"].where(raw["CYP2C9 consensus"].isin(cyp_ok), "Unknown")
    vk_ok = ["A/A", "A/G", "G/G"]
    d["vkorc1"] = raw["VKORC1 -1639 consensus"].where(raw["VKORC1 -1639 consensus"].isin(vk_ok), "Unknown")

    d["site"] = raw["Project Site"]
    d["dose_week"] = pd.to_numeric(raw["Therapeutic Dose of Warfarin"], errors="coerce")
    d["stable"] = raw["Subject Reached Stable Dose of Warfarin"]
    return d

d = clean_iwpc(raw)
d = d[(d["stable"] == 1) & d["dose_week"].notna()].drop(columns="stable").reset_index(drop=True)

print("Usable patients:", len(d))
print("\nMissing fraction per column:")
print(d.isna().mean().round(3))
print("\nDose (mg/week) summary:")
print(d["dose_week"].describe().round(1))

## 4. Design choices: target, imputation, split

In [ ]:
NUM = ["age_decade", "height_cm", "weight_kg", "male"]
BIN = ["inducer", "amiodarone"]
CAT = ["race", "cyp2c9", "vkorc1"]
FEATS = NUM + BIN + CAT

def fit_imputer(train):
    imp = IterativeImputer(max_iter=10, random_state=RANDOM_STATE)
    imp.fit(train[NUM])
    return imp

def apply_imputer(imp, df):
    out = df.copy()
    out[NUM] = imp.transform(df[NUM])
    out["male"] = out["male"].clip(0, 1)
    return out

## 5. Baselines

In [ ]:
def fixed_5mg(df):
    return np.full(len(df), 35.0)

def iwpc_clinical(df):
    asian = (df["race"] == "Asian").astype(float)
    black = (df["race"] == "Black").astype(float)
    miss = (df["race"] == "Unknown").astype(float)
    s = (4.0376 - 0.2546*df["age_decade"] + 0.0118*df["height_cm"] + 0.0134*df["weight_kg"]
         - 0.6752*asian + 0.4060*black + 0.0443*miss
         + 1.2799*df["inducer"] - 0.5695*df["amiodarone"])
    return np.clip(s, 0, None) ** 2

def iwpc_pgx(df):
    vk, cy = df["vkorc1"], df["cyp2c9"]
    s = (5.6044 - 0.2614*df["age_decade"] + 0.0087*df["height_cm"] + 0.0128*df["weight_kg"]
         - 0.8677*(vk == "A/G") - 1.6974*(vk == "A/A") - 0.4854*(vk == "Unknown")
         - 0.5211*(cy == "*1/*2") - 0.9357*(cy == "*1/*3") - 1.0616*(cy == "*2/*2")
         - 1.9206*(cy == "*2/*3") - 2.3312*(cy == "*3/*3") - 0.2188*(cy == "Unknown")
         - 0.1092*(df["race"] == "Asian") - 0.2760*(df["race"] == "Black") - 0.1032*(df["race"] == "Unknown")
         + 1.1816*df["inducer"] - 0.5503*df["amiodarone"])
    return np.clip(s, 0, None) ** 2

## 6. Models

In [ ]:
def make_models():
    enc = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT)],
                            remainder="passthrough", verbose_feature_names_out=False)
    def pipe(reg):
        return Pipeline([("enc", enc), ("reg", reg)])
    ridge = pipe(RidgeCV(alphas=np.logspace(-2, 3, 20)))
    rf = pipe(RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE))
    hgb = pipe(HistGradientBoostingRegressor(max_depth=4, learning_rate=0.05, max_iter=300, random_state=RANDOM_STATE))
    stack = StackingRegressor(
        estimators=[("ridge", ridge), ("rf", rf), ("hgb", hgb)],
        final_estimator=RidgeCV(alphas=np.logspace(-2, 3, 20)), cv=5)
    return {"Ridge (sqrt dose)": ridge, "Random Forest": rf,
            "Gradient Boosting": hgb, "Stacked Ensemble": stack}

## 7. Evaluation metrics

In [ ]:
def dose_metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return {"MAE": mean_absolute_error(y, p), "R2": r2_score(y, p),
            "within20%": np.mean(np.abs(p - y) <= 0.2*y),
            "over>20%": np.mean(p > 1.2*y), "under>20%": np.mean(p < 0.8*y)}

def run_split(train, test):
    imp = fit_imputer(train)                       # fit on TRAIN only
    tr, te = apply_imputer(imp, train), apply_imputer(imp, test)
    ytr = np.sqrt(tr["dose_week"].values)
    results, preds, fitted = {}, {}, {}
    for name, fn in [("Fixed 5 mg/day", fixed_5mg), ("IWPC clinical", iwpc_clinical),
                     ("IWPC pharmacogenetic", iwpc_pgx)]:
        preds[name] = fn(te)
        results[name] = dose_metrics(te["dose_week"], preds[name])
    for name, m in make_models().items():
        m.fit(tr[FEATS], ytr)
        preds[name] = np.clip(m.predict(te[FEATS]), 0, None) ** 2
        results[name] = dose_metrics(te["dose_week"], preds[name])
        fitted[name] = m
    return pd.DataFrame(results).T, preds, fitted, imp, te

## 8. Run 1: hold-out split

In [ ]:
train, test = train_test_split(d, test_size=0.2, random_state=RANDOM_STATE, stratify=d["race"])
print("train:", len(train), " test:", len(test))

res, preds, fitted, imp, te = run_split(train, test)
res.round(3).to_csv("outputs/holdout_results.csv")
res.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(te["dose_week"], preds["Stacked Ensemble"], s=8, alpha=0.4)
ax[0].plot([0, 120], [0, 120], "r--", label="perfect")
ax[0].set_xlim(0, 120); ax[0].set_ylim(0, 120)
ax[0].set_xlabel("Actual stable dose (mg/week)"); ax[0].set_ylabel("Predicted (mg/week)")
ax[0].set_title("Stacked ensemble: predicted vs actual"); ax[0].legend()

res["within20%"].sort_values().plot.barh(ax=ax[1])
ax[1].set_xlabel("Fraction within 20% of true dose"); ax[1].set_title("Ideal-dose rate by model")
plt.tight_layout(); plt.savefig("outputs/holdout_plots.png", dpi=150); plt.show()

## 9. Run 2: unseen hospital sites (harder test)

In [ ]:
rows = []
gkf = GroupKFold(n_splits=5)
for k, (tr_i, te_i) in enumerate(gkf.split(d, groups=d["site"])):
    r, _, _, _, _ = run_split(d.iloc[tr_i], d.iloc[te_i])
    r["fold"] = k
    rows.append(r.reset_index().rename(columns={"index": "model"}))
site_cv = pd.concat(rows)
site_summary = site_cv.groupby("model")[["MAE", "R2", "within20%"]].agg(["mean", "std"]).round(3)
site_summary.to_csv("outputs/unseen_site_results.csv")
site_summary

## 10. Subgroup performance

In [ ]:
te = te.copy()
te["pred_ens"] = preds["Stacked Ensemble"]
te["pred_iwpc"] = preds["IWPC pharmacogenetic"]
te["dose_group"] = pd.cut(te["dose_week"], [0, 21, 49, 1000],
                          labels=["Low (<=21)", "Intermediate (21-49)", "High (>49)"])

def subgroup_table(col):
    rows = []
    for g, s in te.groupby(col, observed=True):
        rows.append({col: g, "n": len(s),
                     "MAE ensemble": mean_absolute_error(s["dose_week"], s["pred_ens"]),
                     "MAE IWPC": mean_absolute_error(s["dose_week"], s["pred_iwpc"]),
                     "within20% ensemble": np.mean(np.abs(s["pred_ens"] - s["dose_week"]) <= 0.2*s["dose_week"])})
    return pd.DataFrame(rows).round(3)

for col in ["race", "vkorc1", "dose_group"]:
    print(subgroup_table(col).to_string(index=False), "\n")

## 11. Which features matter? (permutation importance)

In [ ]:
pi = permutation_importance(fitted["Stacked Ensemble"], te[FEATS], np.sqrt(te["dose_week"]),
                            n_repeats=5, random_state=RANDOM_STATE, scoring="neg_mean_absolute_error")
imp_df = pd.DataFrame({"feature": FEATS, "importance": pi.importances_mean}).sort_values("importance")
imp_df.plot.barh(x="feature", y="importance", legend=False, figsize=(6, 4))
plt.title("Permutation importance (higher = matters more)")
plt.tight_layout(); plt.savefig("outputs/feature_importance.png", dpi=150); plt.show()

## 12. Train the final model on all data and save it

In [ ]:
final_imp = fit_imputer(d)
d_imp = apply_imputer(final_imp, d)
final_model = make_models()["Stacked Ensemble"].fit(d_imp[FEATS], np.sqrt(d_imp["dose_week"]))
joblib.dump({"imputer": final_imp, "model": final_model, "features": FEATS}, "models/starting_dose_model.joblib")

def predict_starting_dose(age_years, height_cm, weight_kg, male, race, cyp2c9, vkorc1, inducer=0, amiodarone=0):
    """race: Asian/Black/White/Unknown | cyp2c9: *1/*1, *1/*2, *1/*3, *2/*2, *2/*3, *3/*3, Unknown
       vkorc1: A/A, A/G, G/G, Unknown | male: 1 or 0"""
    b = joblib.load("models/starting_dose_model.joblib")
    row = pd.DataFrame([{"age_decade": age_years // 10, "height_cm": height_cm, "weight_kg": weight_kg,
                         "male": male, "race": race, "cyp2c9": cyp2c9, "vkorc1": vkorc1,
                         "inducer": inducer, "amiodarone": amiodarone}])
    row = apply_imputer(b["imputer"], row)
    weekly = float(np.clip(b["model"].predict(row[b["features"]]), 0, None)[0] ** 2)
    return {"weekly_mg": round(weekly, 1), "daily_mg": round(weekly / 7, 2)}

# Demo: 72-year-old Asian woman, CYP2C9 *1/*3, VKORC1 A/A (expect a LOW dose)
print(predict_starting_dose(72, 165, 70, 0, "Asian", "*1/*3", "A/A"))
# Same patient with normal-sensitivity genotypes (expect a much HIGHER dose)
print(predict_starting_dose(72, 165, 70, 0, "Asian", "*1/*1", "G/G"))